# Venue comparison — how the consolidated tape relates to its sources

For a flagship pair, this notebook compares the per-venue 1-second close series against the cross-venue **consolidated tape** (`venue='agg'`), looks at the tracking error of each venue vs. the tape, the cross-venue dispersion, and which venues actually contributed to the tape over time (via the `sources_mask` provenance bitmask).

Run `cda-build-dataset` first so there's data. Plots use `matplotlib` (`pip install matplotlib`).

In [ ]:
import pandas as pd
from cryptodata import get_bars, list_symbols
from cryptodata.core.symbols import venue_bitmask_index
from cryptodata.sources.registry import all_sources
from cryptodata.storage.duckdb_views import connect

SYMBOL = 'BTC-USD'   # a flagship pair with a multi-venue consolidated tape

# Window = whatever 1s bars we have for this symbol.
with connect() as con:
    lo, hi, n = con.execute('SELECT MIN(ts_ns), MAX(ts_ns), COUNT(*) FROM bars_1s WHERE symbol = ?', [SYMBOL]).fetchone()
    venues = [r[0] for r in con.execute("SELECT DISTINCT venue FROM bars_1s WHERE symbol = ? AND venue != 'agg' ORDER BY venue", [SYMBOL]).fetchall()]
start, end = pd.Timestamp(lo, unit='ns', tz='UTC'), pd.Timestamp(hi, unit='ns', tz='UTC') + pd.Timedelta(seconds=1)
print(f'{SYMBOL}: {n} 1s bars over {start} .. {end}; per-venue series: {venues}')

In [ ]:
agg = get_bars(SYMBOL, start, end, '1s', sources=['agg'])
per_venue = {v: get_bars(SYMBOL, start, end, '1s', sources=[v]) for v in venues}

wide = pd.concat({v: df['close'] for v, df in per_venue.items()}, axis=1)
wide['agg'] = agg['close']
wide = wide.sort_index()
wide.tail()

### Tracking error: each venue's close vs. the consolidated close, in bps

In [ ]:
te_bps = (wide[venues].sub(wide['agg'], axis=0)).div(wide['agg'], axis=0) * 1e4
print(te_bps.describe().T[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

# cross-venue dispersion per second (high - low across venues), in bps of the mid
rowmin, rowmax = wide[venues].min(axis=1), wide[venues].max(axis=1)
disp_bps = (rowmax - rowmin) / ((rowmin + rowmax) / 2) * 1e4
print('\ncross-venue close dispersion (bps):'); print(disp_bps.describe())

In [ ]:
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    wide[['agg'] + venues].plot(ax=ax[0], lw=0.8); ax[0].set_title(f'{SYMBOL} — 1s close: consolidated tape vs. venues'); ax[0].set_ylabel('price')
    te_bps.plot(ax=ax[1], lw=0.6); ax[1].axhline(0, color='k', lw=0.5); ax[1].set_title('tracking error vs. consolidated (bps)'); ax[1].set_ylabel('bps')
    plt.tight_layout(); plt.show()
except ImportError:
    print('install matplotlib for the plots')

### Provenance: which venues contributed to the tape, second by second

`sources_mask` is a bitmask over the *global* venue→bit index, so it's stable across days. We decode it into a per-venue presence matrix.

In [ ]:
bit = venue_bitmask_index(all_sources())
mask = agg['sources_mask'].astype('int64')
presence = pd.DataFrame({v: ((mask & b) != 0).astype(int) for v, b in bit.items() if v in venues}, index=agg.index)
print('share of seconds each venue contributed to the tape:')
print((presence.mean() * 100).round(1).astype(str) + ' %')
print('\nseconds with N contributing venues:'); print(presence.sum(axis=1).value_counts().sort_index())